# 🐸 Big run: pretrain + fine-tune (for "Save & Run All")

This notebook only contains the big run, so it can run in the background with **Save Version → Save & Run All (Commit)** (the tab may be closed). Afterwards the results are under the version's *Output*: `big/pre/…`, `big/ft/…` with models, sample grids, `loss.png`, `metrics.csv`.

Before: GPU (T4) and Internet on, add the dataset(s) with the folders `wednesday-frogs/` and `frog-photos/` as input.
The parameters are in the code cell right after the setup – adjust them after the speed test.

In [ ]:
import os
REPO = "https://github.com/alexanderulbrich/wednesday.git"
# Always work with absolute paths – that way this cell may run any number of times (first time: clone, afterwards: update).
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
%cd {WORK}
!git clone -q $REPO wednesday 2>/dev/null || git -C wednesday pull -q
%cd {WORK}/wednesday
!pip install -q -e .
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os

def find_dataset(name):
    """The folder name/128 with the images – wherever Kaggle/Colab mounts it.

    The folder directly above 128/ must be called exactly `name`. That works with two separate datasets
    (/kaggle/input/frog-photos/128) just as well as with both zips in one shared dataset
    (/kaggle/input/whatever/frog-photos/128).
    """
    hits = sorted(glob.glob(f"{ROOT}/**/{name}/128", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{name}/128' not found under {ROOT} – dataset added as input? "
                                f"Check the contents with: !find {ROOT} -maxdepth 5 -type d")
    if len(hits) > 1:
        print(f"Warning, several matches for {name}, taking the first: {hits}")
    return hits[0]

if os.path.exists("/kaggle"):
    ROOT, OUT = "/kaggle/input", "/kaggle/working"
else:  # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT = "/content/data", "/content/drive/MyDrive/wednesday-runs"   # runs on Drive -> survive timeouts
    for z in ("wednesday-frogs", "frog-photos"):
        !mkdir -p {ROOT}/{z} && unzip -qo /content/drive/MyDrive/{z}.zip -d {ROOT}/{z}

MEMES, PHOTOS = find_dataset("wednesday-frogs"), find_dataset("frog-photos")
RES = 64   # training happens at 64 px, the source are the sharper 128 px images
for d in (MEMES, PHOTOS):
    print(d, len(os.listdir(d)), "images")

In [ ]:
# ---- Parameters (times in minutes; the number of steps follows from the measured speed) ----
BASE = 64        # network size: 64 ≈ 18M parameters, 48 ≈ 10M, 32 ≈ 4.5M
PRE_MIN = 180    # pretraining on photos + memes
FT_MIN = 30      # fine-tune on memes only, keeps the state with the best meme validation loss
RUN = f"{OUT}/big"
print(BASE, PRE_MIN, FT_MIN, RUN)

## Stage 1: pretrain

In [ ]:
!python train.py --data {PHOTOS},{MEMES} --res {RES} --base {BASE} --minutes {PRE_MIN} --steps 1000000 --lr 2e-4 --warmup 1000 --batch 64 --out {RUN}/pre --sample-every 2000 --val-every 1000 --save-every 2000 --keep-best val

## Stage 2: fine-tune (best state by meme validation → `best_frog.pt`)

In [ ]:
!python train.py --init {RUN}/pre/last.pt --data {MEMES} --res {RES} --minutes {FT_MIN} --steps 1000000 --lr 1e-4 --warmup 200 --batch 64 --out {RUN}/ft --sample-every 500 --val-every 250 --save-every 1000 --keep-best wednesday-frogs

## Result

In [ ]:
from IPython.display import Image, display
for stage, ckpt in (("pre", "frog.pt"), ("ft", "best_frog.pt")):
    !python generate.py --ckpt {RUN}/{stage}/{ckpt} --n 16 --grid --size 128 --seed 42 --no-open --out {RUN}/{stage}/generated
    print(stage); display(Image(f"{RUN}/{stage}/loss.png"))
    display(Image(sorted(glob.glob(f"{RUN}/{stage}/generated/*.png"))[-1]))